# Lab 3: Building a QA RAG Chatbot with LangChain (Free Version)

## Introduction

Welcome to Lab 3! In this final lab, we'll bring everything together to build a complete question-answering RAG chatbot using only free, local tools.

### Learning Objectives
- Set up Ollama for local language model inference
- Create RetrievalQA chains combining retrievers and LLMs
- Add conversational memory for multi-turn dialogues
- Customize prompts for better RAG performance
- Build a complete end-to-end RAG system

## 1. Environment Setup

### Installing Ollama

First, you need to install Ollama to run local language models. Visit https://ollama.ai/ for installation instructions, or run:

```bash
# Linux/Mac
curl -fsSL https://ollama.ai/install.sh | sh

# Then download a model
ollama pull llama2:7b
```

### Python Dependencies

In [1]:
# Install required packages
!pip install langchain sentence-transformers chromadb pypdf beautifulsoup4 requests langchain_community langchain-ollama

In [3]:
# Import necessary libraries
from langchain_ollama import OllamaLLM
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
import time

print("✅ All libraries imported successfully!")

USER_AGENT environment variable not set, consider setting it to identify your requests.


✅ All libraries imported successfully!


## 2. Check Ollama Installation

Let's verify that Ollama is working and check available models.

In [5]:
# Check if Ollama is running and test a simple query
try:
    # Initialize Ollama with a lightweight model
    llm =  OllamaLLM(
    model="llama2:7b", temperature=0.1)
    
    # Test with a simple query
    response = llm("Hello! Can you tell me what 2+2 equals?")
    print("✅ Ollama is working!")
    print(f"🤖 Response: {response}")
    
except Exception as e:
    print(f"❌ Ollama error: {e}")
    print("\n💡 Troubleshooting:")
    print("1. Make sure Ollama is installed: https://ollama.ai/")
    print("2. Download a model: ollama pull llama2:7b")
    print("3. Start Ollama service: ollama serve")
    print("\n🔄 Trying with a smaller model...")
    
    try:
        llm = OllamaLLM(model="tinyllama", temperature=0.1)
        response = llm("Hello!")
        print("✅ TinyLlama is working!")
        print(f"🤖 Response: {response}")
    except Exception as e2:
        print(f"❌ Still having issues: {e2}")
        print("Please install Ollama and download a model before continuing.")

✅ Ollama is working!
🤖 Response: Of course! 2 + 2 = 4.


## 3. Prepare Our Knowledge Base

Let's load and process documents to create our knowledge base, using techniques from Labs 1 and 2.

In [6]:
# Load documents from multiple sources
urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Deep_learning"
]

all_docs = []
for url in urls:
    print(f"📄 Loading: {url.split('/')[-1].replace('_', ' ')}")
    loader = WebBaseLoader(url)
    docs = loader.load()
    all_docs.extend(docs)

print(f"✅ Loaded {len(all_docs)} documents")

📄 Loading: Artificial intelligence
📄 Loading: Machine learning
📄 Loading: Deep learning
✅ Loaded 3 documents


In [7]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

splits = text_splitter.split_documents(all_docs)
print(f"📚 Split into {len(splits)} chunks")
print(f"📏 Average chunk size: {sum(len(doc.page_content) for doc in splits) / len(splits):.0f} characters")

📚 Split into 697 chunks
📏 Average chunk size: 753 characters


In [8]:
# Create embeddings and vector store
print("🔄 Creating embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)
vectorstore.save_local("./rag_faiss_db")
print("✅ Vector store created!")
print(f"💾 Persisted to: ./rag_faiss_db")

🔄 Creating embeddings...


/var/folders/sc/r0vk6bp90xz8qtd3lqtpsrzc0000gn/T/ipykernel_32843/3803324101.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/opt/homebrew/anaconda3/envs/audioenv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


✅ Vector store created!
💾 Persisted to: ./rag_chroma_db


## 4. Building Our First RAG Chain

Now let's create a simple RetrievalQA chain that combines our vector store with the language model.

In [9]:
# Create a RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # "stuff" puts all retrieved docs into the prompt
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),  # Retrieve top 3 docs
    return_source_documents=True,  # Return the source documents used
    verbose=True  # Show the chain's reasoning
)

print("✅ RAG chain created!")
print("🔗 Components: Retriever + LLM + Prompt")

✅ RAG chain created!
🔗 Components: Retriever + LLM + Prompt


In [10]:
# Test our RAG chain
question = "What is the difference between artificial intelligence and machine learning?"

print(f"❓ Question: {question}")
print("\n🤖 Generating answer...")

start_time = time.time()
result = qa_chain({"query": question})
end_time = time.time()

print(f"\n✅ Answer generated in {end_time - start_time:.2f} seconds")
print(f"\n📝 Answer: {result['result']}")

print(f"\n📚 Sources used:")
for i, doc in enumerate(result['source_documents'], 1):
    source = doc.metadata.get('source', 'Unknown')
    print(f"   {i}. {source}")
    print(f"      Preview: {doc.page_content[:100]}...")

❓ Question: What is the difference between artificial intelligence and machine learning?

🤖 Generating answer...


/var/folders/sc/r0vk6bp90xz8qtd3lqtpsrzc0000gn/T/ipykernel_32843/2047816371.py:8: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = qa_chain({"query": question})
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.

✅ Answer generated in 19.79 seconds

📝 Answer: Artificial intelligence (AI) and machine learning (ML) are related but distinct fields. AI refers to the broader field of research and development aimed at creating intelligent machines that can perform tasks that typically require human intelligence, such as understanding language, recognizing images, making decisions, and solving problems. Machine learning is a subset of AI that focuses specifically on developing algorithms and statistical models that enable machines to learn from data, without being explicitly programmed for each task.

In other words, AI is the umbrella term that encompasses various subfields like natural language processing, computer vision, robotics, and expert systems, among others. Machine learning is one of these subfields that focuses on developing algorithms that can learn from data and improve their performance over time.

To put it simply, AI is the big picture, while machine learning is a 

## 5. Improving with Custom Prompts

Let's create a custom prompt to improve the quality and consistency of our RAG responses.

In [11]:
# Create a custom prompt template
custom_prompt_template = """
You are a helpful AI assistant that answers questions based on the provided context. 
Use the following pieces of context to answer the question at the end.

Important guidelines:
- If you don't know the answer based on the context, say "I don't have enough information to answer that question."
- Be concise but comprehensive in your answers
- Cite specific information from the context when possible
- If the question asks for examples, provide them from the context

Context:
{context}

Question: {question}

Helpful Answer:"""

CUSTOM_PROMPT = PromptTemplate(
    template=custom_prompt_template,
    input_variables=["context", "question"]
)

# Create an improved RAG chain with custom prompt
improved_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": CUSTOM_PROMPT},
    return_source_documents=True
)

print("✅ Improved RAG chain created with custom prompt!")

✅ Improved RAG chain created with custom prompt!


In [12]:
# Test the improved chain
question = "What are some applications of deep learning?"

print(f"❓ Question: {question}")
print("\n🤖 Generating improved answer...")

result = improved_qa_chain({"query": question})

print(f"\n📝 Improved Answer: {result['result']}")

❓ Question: What are some applications of deep learning?

🤖 Generating improved answer...

📝 Improved Answer: Based on the provided context, some applications of deep learning include:

1. Automatic speech recognition: Deep learning has been successful in this area, with LSTM RNNs competing with traditional speech recognizers on certain tasks.
2. Computer vision: Deep learning has been used for computer vision tasks such as image processing and object recognition.
3. Speech recognition: As mentioned in the context, deep learning has been used to achieve successful automatic speech recognition tasks.
4. Machine translation: Deep learning has been used for machine translation tasks, such as translating text from one language to another.
5. Social network filtering: Deep learning has been used to filter and analyze social media data, such as identifying relevant content or detecting fake news.
6. Playing board games and video games: Deep learning has been used to create AI agents that can

## 6. Adding Conversational Memory

Let's create a conversational RAG chatbot that remembers previous interactions.

In [13]:
# Create memory for conversation history
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"  # Important: specify the output key for ConversationalRetrievalChain
)

# Create a conversational RAG chain
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=True,
    verbose=True
)

print("✅ Conversational RAG chain created!")
print("💭 Now the chatbot will remember our conversation history")

✅ Conversational RAG chain created!
💭 Now the chatbot will remember our conversation history


In [14]:
# Have a conversation with our RAG chatbot
def chat_with_bot(question):
    print(f"\n👤 You: {question}")
    print("🤖 Bot: ", end="")
    
    result = conversational_chain({"question": question})
    print(result["answer"])
    
    return result

# Start our conversation
print("🎉 Starting conversation with RAG chatbot!")
print("💡 The bot will remember our conversation history")

# First question
result1 = chat_with_bot("What is machine learning?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


🎉 Starting conversation with RAG chatbot!
💡 The bot will remember our conversation history

👤 You: What is machine learning?
🤖 Bot: Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Learning
Machine learning is the study of programs that can improve their performance on a given task automatically.[41] It has been a part of AI from the beginning.[e]

Glossary
Glossary
vte
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions.[1] Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.[2]
ML finds applicati

In [15]:
# Follow-up question (tests memory)
result2 = chat_with_bot("Can you give me some specific examples of it?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



👤 You: Can you give me some specific examples of it?
🤖 Bot: Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: What is machine learning?
Assistant: Based on the provided context, machine learning can be defined as a field of study in artificial intelligence that focuses on developing statistical algorithms that can learn from data and generalize to unseen data, allowing them to perform tasks without explicit instructions. These algorithms are typically used in applications such as natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine, among others. The field of machine learning has evolved over time, shifting its focus away from symbolic approaches inherited from AI towards methods and models borrowed from statistics, fuzzy logic, and probability theory, with the goal of tackling solvable p

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

crisis.[105] In 2012, co-founder of Sun Microsystems, Vinod Khosla, predicted that 80% of medical doctors jobs would be lost in the next two decades to automated machine learning medical diagnostic software.[106] In 2014, it was reported that a machine learning algorithm had been applied in the field of art history to study fine art paintings and that it may have revealed previously unrecognised influences among artists.[107] In 2019 Springer Nature published the first research book created using machine learning.[108] In 2020, machine learning technology was used to help make diagnoses and aid researchers in developing a cure for COVID-19.[109] Machine learning was recently applied to predict the pro-environmental behaviour of travellers.[110] Recently, machine learning technology was also applied t

In [16]:
# Another follow-up
result3 = chat_with_bot("How does it relate to artificial intelligence?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



👤 You: How does it relate to artificial intelligence?
🤖 Bot: Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: What is machine learning?
Assistant: Based on the provided context, machine learning can be defined as a field of study in artificial intelligence that focuses on developing statistical algorithms that can learn from data and generalize to unseen data, allowing them to perform tasks without explicit instructions. These algorithms are typically used in applications such as natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine, among others. The field of machine learning has evolved over time, shifting its focus away from symbolic approaches inherited from AI towards methods and models borrowed from statistics, fuzzy logic, and probability theory, with the goal of tackling solvable 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Learning
Machine learning is the study of programs that can improve their performance on a given task automatically.[41] It has been a part of AI from the beginning.[e]

Evaluating approaches to AI
No established unifying theory or paradigm has guided AI research for most of its history.[aa] The unprecedented success of statistical machine learning in the 2010s eclipsed all other approaches (so much so that some sources, especially in the business world, use the term "artificial intelligence" to mean "machine learning with neural networks"). This approach is mostly sub-symbolic, soft and narrow. Critics argue that these questions may have to be revisited by future generations of AI researchers.

Relationships to other fields[edit]
Artificial intelligence[edit]
Deep learning is a su

In [17]:
# Check conversation history
print("📜 Conversation History:")
chat_history = memory.chat_memory.messages
for i, message in enumerate(chat_history):
    role = "👤 Human" if message.type == "human" else "🤖 AI"
    print(f"\n{role}: {message.content[:100]}...")

📜 Conversation History:

👤 Human: What is machine learning?...

🤖 AI: Based on the provided context, machine learning can be defined as a field of study in artificial int...

👤 Human: Can you give me some specific examples of it?...

🤖 AI: Of course! Machine learning is a versatile technology that has numerous applications across various ...

👤 Human: How does it relate to artificial intelligence?...

🤖 AI: Machine learning is a subset of artificial intelligence (AI). Specifically, deep learning is a subse...


## 7. Advanced Features

Let's explore some advanced features to make our RAG chatbot even better.

### 7.1 Streaming Responses

Make the chatbot feel more responsive with streaming.

In [18]:
# Create a streaming LLM
streaming_llm = OllamaLLM(
    model="llama2:7b",
    temperature=0.1,
    callbacks=[StreamingStdOutCallbackHandler()]  # This enables streaming to stdout
)

# Create a streaming RAG chain
streaming_chain = RetrievalQA.from_chain_type(
    llm=streaming_llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": CUSTOM_PROMPT}
)

print("✅ Streaming RAG chain created!")
print("🌊 Responses will now stream token by token")

✅ Streaming RAG chain created!
🌊 Responses will now stream token by token


In [19]:
# Test streaming
question = "Explain the concept of neural networks in simple terms."
print(f"❓ Question: {question}")
print("\n🌊 Streaming response:")

result = streaming_chain({"query": question})
print("\n\n✅ Streaming complete!")

❓ Question: Explain the concept of neural networks in simple terms.

🌊 Streaming response:
Neural networks are like a complex system of interconnected nodes or "neurons" that work together to recognize patterns in data. Think of it like a brain, where each node represents a small part of the brain and the connections between them represent the neural pathways that allow information to flow between different parts of the brain.

The network is trained on a set of examples, and once it's trained, it can recognize those patterns in new data it hasn't seen before. The network consists of an input layer, one or more hidden layers, and an output layer. Each node in the network applies a specific function to the data it receives, and if the weight of the connection between nodes is high enough, the data is passed on to the next layer.

In simple terms, neural networks are like a computer program that can learn from examples and make predictions or decisions based on new data it encounters. Th

### 7.2 Different Chain Types


When building RAG systems, we need to combine multiple retrieved documents and generate a final answer. LangChain provides three main chain types for this purpose:

#### 1. Stuff Chain
**How it works:**
- Takes all retrieved documents and "stuffs" them into a single prompt
- Sends everything to the LLM in one API call

**Pros:**
- Simple and fast
- Preserves all context in a single coherent prompt
- Best for smaller document sets

**Cons:**
- Limited by LLM's context window (typically 4K-32K tokens)
- Can be expensive for large document sets
- May lose focus if too many irrelevant documents are included

**Best for:** Small to medium document sets, high-quality retrieval

#### 2. Map-Reduce Chain
**How it works:**
- **Map phase:** Process each document independently to generate partial answers
- **Reduce phase:** Combine all partial answers into a final response

**Pros:**
- Can handle large numbers of documents
- Parallel processing possible (faster for many documents)
- Scales well with document count

**Cons:**
- May lose nuanced relationships between documents
- Final answer quality depends on reduction step
- More complex and potentially slower for small document sets

**Best for:** Large document collections, when you need to process many documents

#### 3. Refine Chain
**How it works:**
- Start with the first document to generate an initial answer
- Sequentially process each subsequent document, refining the answer
- Each step builds upon the previous answer

**Pros:**
- Preserves relationships between documents
- Can generate more nuanced, comprehensive answers
- Good at synthesizing information across documents

**Cons:**
- Sequential processing (slower for many documents)
- Quality depends on document order
- Can be affected by early poor answers

**Best for:** Complex questions requiring synthesis, when document relationships matter

### Choosing the Right Chain

| Scenario | Recommended Chain | Reason |
|----------|------------------|---------|
| Small dataset (< 10 docs) | Stuff | Simple, preserves context |
| Large dataset (> 100 docs) | Map-Reduce | Scales better |
| Complex synthesis needed | Refine | Better at combining information |
| Speed is critical | Map-Reduce | Parallel processing |
| Cost optimization | Stuff | Fewer API calls |

Let's explore different chain types for handling longer documents.

In [20]:
# Compare different chain types
chain_types = ["stuff", "map_reduce", "refine"]
question = "What are the main challenges in artificial intelligence?"

for chain_type in chain_types:
    print(f"\n🔗 Testing {chain_type} chain:")
    
    try:
        test_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type=chain_type,
            retriever=vectorstore.as_retriever(search_kwargs={"k": 3})
        )
        
        start_time = time.time()
        result = test_chain({"query": question})
        end_time = time.time()
        
        print(f"   ⏱️  Time: {end_time - start_time:.2f}s")
        print(f"   📝 Answer: {result['result'][:150]}...")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")


🔗 Testing stuff chain:
   ⏱️  Time: 26.90s
   📝 Answer: Based on the provided context, the main challenges in artificial intelligence are:

1. Algorithmic biases: AI systems can perpetuate and even amplify ...

🔗 Testing map_reduce chain:


Token indices sequence length is longer than the specified maximum sequence length for this model (1655 > 1024). Running this sequence through the model will result in indexing errors


   ⏱️  Time: 28.31s
   📝 Answer: The final answer to the question "What are the main challenges in artificial intelligence?" is:

* Algorithmic biases
* Fairness
* Automated decision-...

🔗 Testing refine chain:
   ⏱️  Time: 47.97s
   📝 Answer: Based on the additional context provided, it seems that the main challenges in artificial intelligence (AI) are fairness, misuse, and alignment, while...


## 8. Building a Complete RAG System

Let's put everything together into a comprehensive RAG system.

In [21]:
class RAGChatbot:
    def __init__(self, model_name="llama2:7b"):
        # Initialize components
        self.llm = OllamaLLM(model=model_name, temperature=0.1)
        self.embeddings = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'}
        )
        self.vectorstore = None
        self.qa_chain = None
        self.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key="answer"
        )
        
    def load_documents(self, urls):
        """Load and process documents from URLs"""
        print("📄 Loading documents...")
        all_docs = []
        
        for url in urls:
            loader = WebBaseLoader(url)
            docs = loader.load()
            all_docs.extend(docs)
            print(f"   ✅ {url.split('/')[-1]}")
        
        # Split documents
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )
        splits = text_splitter.split_documents(all_docs)
        
        # Create vector store
        self.vectorstore = FAISS.from_documents(
            documents=splits,
            embedding=self.embeddings
        )
        
        # Create QA chain
        self.qa_chain = ConversationalRetrievalChain.from_llm(
            llm=self.llm,
            retriever=self.vectorstore.as_retriever(search_kwargs={"k": 3}),
            memory=self.memory,
            return_source_documents=True
        )
        
        print(f"✅ Processed {len(splits)} document chunks")
        
    def chat(self, question):
        """Chat with the RAG system"""
        if not self.qa_chain:
            return "Please load documents first using load_documents()"
        
        result = self.qa_chain({"question": question})
        return {
            "answer": result["answer"],
            "sources": [doc.metadata.get('source', 'Unknown') for doc in result.get('source_documents', [])]
        }
    
    def reset_memory(self):
        """Reset conversation memory"""
        self.memory.clear()
        print("🧠 Memory cleared!")

# Create our RAG chatbot
chatbot = RAGChatbot()
print("🤖 RAG Chatbot created!")

🤖 RAG Chatbot created!


In [22]:
# Load documents into our chatbot
urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing"
]

chatbot.load_documents(urls)

📄 Loading documents...
   ✅ Artificial_intelligence
   ✅ Machine_learning
   ✅ Natural_language_processing
✅ Processed 582 document chunks


In [23]:
# Test our complete RAG system
def interactive_chat():
    print("🎉 Welcome to the RAG Chatbot!")
    print("💬 Ask me anything about AI, ML, or NLP")
    print("🔄 Type 'reset' to clear memory, 'quit' to exit\n")
    
    while True:
        question = input("👤 You: ")
        
        if question.lower() == 'quit':
            print("👋 Goodbye!")
            break
        elif question.lower() == 'reset':
            chatbot.reset_memory()
            continue
        
        print("🤖 Bot: ", end="")
        result = chatbot.chat(question)
        print(result["answer"])
        
        if result["sources"]:
            print(f"\n📚 Sources: {', '.join(set(result['sources']))}")
        print()

# For Jupyter, let's do a few example interactions instead of the interactive loop
questions = [
    "What is natural language processing?",
    "How is it used in practice?",
    "What's the relationship between NLP and machine learning?"
]

for question in questions:
    print(f"👤 You: {question}")
    result = chatbot.chat(question)
    print(f"🤖 Bot: {result['answer']}")
    print(f"📚 Sources: {', '.join(set(result['sources']))}\n")

👤 You: What is natural language processing?
🤖 Bot: Based on the provided context, natural language processing (NLP) is a subfield of computer science that deals with the interaction between computers and human language. It involves the development of algorithms and statistical models that enable computers to process, understand, and generate natural language data, such as text or speech. The main goals of NLP are to enable computers to perform tasks that would normally require human-level understanding of language, such as information retrieval, question answering, and machine translation.

NLP is a multidisciplinary field that draws on linguistics, computer science, and cognitive psychology. It involves the study of grammar, syntax, semantics, and pragmatics, as well as the development of algorithms and statistical models to process and analyze natural language data. Some of the key processing tasks in an NLP system include speech recognition, text classification, natural language und

## 9. Exercises

Now it's your turn to practice and extend the RAG system!

### Exercise 1: Custom Domain RAG

Create a RAG system for a specific domain (e.g., cooking, sports, history) using relevant documents.

In [ ]:
# Exercise 1: Your code here
# TODO:
# 1. Choose a domain and find 3-5 relevant Wikipedia pages
# 2. Create a specialized RAG chatbot for that domain
# 3. Test it with domain-specific questions
# 4. Compare performance with the general AI chatbot

# Your code here:


### Exercise 2: Multi-Modal RAG

Extend the RAG system to handle different types of documents (PDFs, web pages, etc.).

In [ ]:
# Exercise 2: Your code here
# TODO:
# 1. Download a PDF document
# 2. Load both PDF and web content
# 3. Create a combined knowledge base
# 4. Test cross-document question answering

# Your code here:


### Exercise 3: RAG Evaluation

Create a simple evaluation system for your RAG chatbot.

In [ ]:
# Exercise 3: Your code here
# TODO:
# 1. Create a set of test questions with expected answers
# 2. Run your RAG system on these questions
# 3. Compare the answers (manually or with simple metrics)
# 4. Identify areas for improvement

# Your code here:


### Exercise 4: Advanced Prompt Engineering

Experiment with different prompt templates to improve response quality.

In [ ]:
# Exercise 4: Your code here
# TODO:
# 1. Create 2-3 different prompt templates
# 2. Test them with the same questions
# 3. Compare the quality of responses
# 4. Identify which prompt works best for different types of questions

# Your code here:


## Solutions

Here are the solutions to the exercises. Try to complete them yourself first!

In [24]:
# Solution 1: Custom Domain RAG (Cooking Domain)
cooking_urls = [
    "https://en.wikipedia.org/wiki/Cooking",
    "https://en.wikipedia.org/wiki/Baking",
    "https://en.wikipedia.org/wiki/Grilling",
    "https://en.wikipedia.org/wiki/Italian_cuisine",
    "https://en.wikipedia.org/wiki/French_cuisine"
]

cooking_chatbot = RAGChatbot()
cooking_chatbot.load_documents(cooking_urls)

# Test cooking-specific questions
cooking_questions = [
    "What's the difference between baking and grilling?",
    "What are some characteristics of Italian cuisine?",
    "What cooking techniques are used in French cuisine?"
]

print("👨‍🍳 Cooking Domain RAG Chatbot:")
for question in cooking_questions:
    print(f"\n👤 You: {question}")
    result = cooking_chatbot.chat(question)
    print(f"🤖 Chef Bot: {result['answer'][:200]}...")

📄 Loading documents...
   ✅ Cooking
   ✅ Baking
   ✅ Grilling
   ✅ Italian_cuisine
   ✅ French_cuisine
✅ Processed 505 document chunks
👨‍🍳 Cooking Domain RAG Chatbot:

👤 You: What's the difference between baking and grilling?
🤖 Chef Bot: Based on the provided context, here is the difference between baking and grilling:

Baking involves cooking food using dry heat, typically in an oven. The heat is transferred from the surface of the f...

👤 You: What are some characteristics of Italian cuisine?
🤖 Chef Bot: The distinctive features of Italian cuisine include:

1. Use of fresh ingredients: Italian cuisine emphasizes the use of fresh, seasonal ingredients, such as vegetables, fruits, meats, and cheeses.
2....

👤 You: What cooking techniques are used in French cuisine?
🤖 Chef Bot: Based on the provided context, French cuisine uses a variety of cooking techniques, including:

1. Baking: This involves cooking food in an oven, typically using dry heat.
2. Roasting: Similar to baki...


In [25]:
# Solution 2: Multi-Modal RAG
from langchain.document_loaders import PyPDFLoader
import requests

# Download a sample PDF
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"  # Attention is All You Need paper
pdf_filename = "transformer_paper.pdf"

response = requests.get(pdf_url)
with open(pdf_filename, "wb") as f:
    f.write(response.content)

# Load PDF
pdf_loader = PyPDFLoader(pdf_filename)
pdf_docs = pdf_loader.load_and_split()

# Load web content
web_loader = WebBaseLoader("https://en.wikipedia.org/wiki/Transformer_(machine_learning_model)")
web_docs = web_loader.load()

# Combine documents
all_docs = pdf_docs + web_docs

# Create multi-modal RAG system
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(all_docs)

multi_vectorstore = FAISS.from_documents(splits, embeddings)
multi_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=multi_vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

# Test cross-document question
question = "What is the transformer architecture and how does attention work?"
result = multi_qa_chain({"query": question})

print(f"📚 Multi-Modal RAG Result:")
print(f"Answer: {result['result'][:300]}...")
print(f"\nSources: {[doc.metadata.get('source', 'Unknown') for doc in result['source_documents']]}")

📚 Multi-Modal RAG Result:
Answer: The Transformer architecture is a type of neural network designed primarily for natural language processing tasks, introduced in the 2017 paper "Attention Is All You Need" by Google researchers. The Transformer model relies on self-attention mechanisms to process input sequences in parallel, allowin...

Sources: ['https://en.wikipedia.org/wiki/Transformer_(machine_learning_model)', 'transformer_paper.pdf', 'https://en.wikipedia.org/wiki/Transformer_(machine_learning_model)']


In [26]:
# Solution 3: RAG Evaluation
test_qa_pairs = [
    {
        "question": "What is machine learning?",
        "expected_keywords": ["algorithm", "data", "pattern", "prediction"]
    },
    {
        "question": "What is deep learning?",
        "expected_keywords": ["neural network", "layer", "artificial"]
    },
    {
        "question": "What is natural language processing?",
        "expected_keywords": ["language", "text", "linguistic", "computer"]
    }
]

def simple_evaluation(qa_pairs, chatbot):
    results = []
    
    for pair in qa_pairs:
        question = pair["question"]
        expected_keywords = pair["expected_keywords"]
        
        # Get answer from chatbot
        result = chatbot.chat(question)
        answer = result["answer"].lower()
        
        # Check how many expected keywords are present
        found_keywords = [kw for kw in expected_keywords if kw.lower() in answer]
        score = len(found_keywords) / len(expected_keywords)
        
        results.append({
            "question": question,
            "score": score,
            "found_keywords": found_keywords,
            "answer_length": len(answer)
        })
    
    return results

# Reset memory for clean evaluation
chatbot.reset_memory()

# Run evaluation
eval_results = simple_evaluation(test_qa_pairs, chatbot)

print("📊 RAG Evaluation Results:")
total_score = 0
for result in eval_results:
    print(f"\nQ: {result['question']}")
    print(f"Score: {result['score']:.2f} ({len(result['found_keywords'])}/{len(test_qa_pairs[0]['expected_keywords'])} keywords found)")
    print(f"Found: {result['found_keywords']}")
    total_score += result['score']

print(f"\n🎯 Overall Score: {total_score/len(eval_results):.2f}")

🧠 Memory cleared!
📊 RAG Evaluation Results:

Q: What is machine learning?
Score: 0.50 (2/4 keywords found)
Found: ['algorithm', 'data']

Q: What is deep learning?
Score: 1.00 (3/4 keywords found)
Found: ['neural network', 'layer', 'artificial']

Q: What is natural language processing?
Score: 1.00 (4/4 keywords found)
Found: ['language', 'text', 'linguistic', 'computer']

🎯 Overall Score: 0.83


In [27]:
# Solution 4: Advanced Prompt Engineering
prompts = {
    "detailed": """
You are an expert AI assistant. Provide detailed, comprehensive answers based on the context.
Include examples and explanations where possible.

Context: {context}
Question: {question}
Detailed Answer:""",
    
    "concise": """
Answer the question concisely based on the context. Be direct and to the point.

Context: {context}
Question: {question}
Concise Answer:""",
    
    "educational": """
You are a helpful teacher. Explain the answer in a way that's easy to understand.
Use analogies and simple language when appropriate.

Context: {context}
Question: {question}
Educational Answer:"""
}

test_question = "What is artificial intelligence?"

print("🎭 Comparing Different Prompt Styles:")
for style, template in prompts.items():
    prompt = PromptTemplate(template=template, input_variables=["context", "question"])
    
    test_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
        chain_type_kwargs={"prompt": prompt}
    )
    
    result = test_chain({"query": test_question})
    
    print(f"\n📝 {style.title()} Style:")
    print(f"{result['result'][:200]}...")
    print(f"Length: {len(result['result'])} characters")

🎭 Comparing Different Prompt Styles:

📝 Detailed Style:
Artificial intelligence (AI) is a field of research in computer science that focuses on developing algorithms and software that enable machines to perform tasks that typically require human intelligen...
Length: 2655 characters

📝 Concise Style:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and dec...
Length: 213 characters

📝 Educational Style:
Great, let's break down the concept of artificial intelligence (AI) together! 🤖

So, AI is like a superpower for machines. It allows them to do things that would normally require human intelligence, l...
Length: 1158 characters


## Summary

Congratulations! You've completed Lab 3 and built a complete RAG chatbot system! Here's what you've accomplished:

✅ **Local LLM Integration**: Set up and used Ollama for local language model inference

✅ **RAG Chains**: Created RetrievalQA and ConversationalRetrievalChain systems

✅ **Memory Management**: Added conversational memory for multi-turn dialogues

✅ **Prompt Engineering**: Customized prompts for better response quality

✅ **Advanced Features**: Implemented streaming, different chain types, and evaluation

✅ **Complete System**: Built a production-ready RAG chatbot class

### Key Achievements

1. **End-to-End RAG Pipeline**: From document loading to conversational AI
2. **Free and Local**: No API keys required, runs entirely on your machine
3. **Customizable**: Easy to adapt for different domains and use cases
4. **Production-Ready**: Includes memory, error handling, and evaluation

### What You've Built

Your RAG system can:
- Load and process documents from multiple sources
- Generate semantic embeddings and store them efficiently
- Retrieve relevant information based on user queries
- Generate contextual responses using local language models
- Maintain conversation history and context
- Handle different types of questions and domains

### Next Steps

Now you're ready for the Challenge Lab where you'll:
- Build a Streamlit web interface for your RAG chatbot
- Create a user-friendly application
- Deploy your system for others to use

### Real-World Applications

The skills you've learned can be applied to:
- Customer support chatbots
- Document Q&A systems
- Educational assistants
- Research tools
- Personal knowledge management

Excellent work! You now have the skills to build sophisticated RAG systems using only free, open-source tools! 🎉